In [3]:
# Step 2: Load Pretrained Word Embedding Model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dim



In [4]:
#Step 3: Connect to Qdrant and Create Collection
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient("http://localhost:6333")  # or Qdrant Cloud URL

client.recreate_collection(
    collection_name="news_vectors",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)



C:\Users\admin\AppData\Local\Temp\ipykernel_2788\3761271622.py:7: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [ ]:
import pandas as pd
import uuid

# Example: Fake and Real News Dataset
df = pd.read_csv("clmentbisaillon/fake-and-real-news-dataset")  # columns: ['title', 'text', 'label']

df["content"] = df["title"] + " " + df["text"]
vectors = model.encode(df["content"].tolist())

payloads = df[["title", "label"]].to_dict(orient="records")
ids = [str(uuid.uuid4()) for _ in range(len(df))]

client.upload_collection(
    collection_name="news_vectors",
    vectors=vectors.tolist(),
    payload=payloads,
    ids=ids
)


In [ ]:
import pandas as pd
import uuid

# Example: Fake and Real News Dataset
df = pd.read_csv("news_dataset.csv")  # columns: ['title', 'text', 'label']

df["content"] = df["title"] + " " + df["text"]
vectors = model.encode(df["content"].tolist())

payloads = df[["title", "label"]].to_dict(orient="records")
ids = [str(uuid.uuid4()) for _ in range(len(df))]

client.upload_collection(
    collection_name="news_vectors",
    vectors=vectors.tolist(),
    payload=payloads,
    ids=ids
)


In [ ]:
def classify_article(text, k=5):
    vector = model.encode([text])[0]
    results = client.search(
        collection_name="news_vectors",
        query_vector=vector,
        limit=k
    )

    votes = [r.payload['label'] for r in results]
    prediction = max(set(votes), key=votes.count)
    return prediction, votes
